Define the network

In [1]:
import pypowsybl as pp
import pandas as pd

# Create an empty network
network = pp.network.create_empty("Simple_3Node_Network")

# Create substations
stations = pd.DataFrame.from_records(index='id', data=[
    {'id': 'Sub_A', 'country': 'SE'},
    {'id': 'Sub_B', 'country': 'SE'},
    {'id': 'Sub_C', 'country': 'SE'}
])
network.create_substations(stations)

# Create voltage levels (400kV)
voltage_levels = pd.DataFrame.from_records(index='id', data=[
    {'id': 'VL_A', 'substation_id': 'Sub_A', 'nominal_v': 400.0, 'topology_kind': 'BUS_BREAKER'},
    {'id': 'VL_B', 'substation_id': 'Sub_B', 'nominal_v': 400.0, 'topology_kind': 'BUS_BREAKER'},
    {'id': 'VL_C', 'substation_id': 'Sub_C', 'nominal_v': 400.0, 'topology_kind': 'BUS_BREAKER'}
])
network.create_voltage_levels(voltage_levels)

# Create buses
buses = pd.DataFrame.from_records(index='id', data=[
    {'id': 'Bus_A', 'voltage_level_id': 'VL_A'},
    {'id': 'Bus_B', 'voltage_level_id': 'VL_B'},
    {'id': 'Bus_C', 'voltage_level_id': 'VL_C'}
])                  
network.create_buses(buses)

# Create lines (A-B, A-C, B-C)
network.create_lines(id='LINE-AB', voltage_level1_id='VL_A', bus1_id='Bus_A',
                     voltage_level2_id='VL_B', bus2_id='Bus_B',
                     b1=1e-6, b2=1e-6, g1=0, g2=0, r=0.5, x=10)
network.create_lines(id='LINE-AC', voltage_level1_id='VL_A', bus1_id='Bus_A',
                     voltage_level2_id='VL_C', bus2_id='Bus_C',
                     b1=1e-6, b2=1e-6, g1=0, g2=0, r=0.5, x=10)
network.create_lines(id='LINE-BC', voltage_level1_id='VL_B', bus1_id='Bus_B',
                     voltage_level2_id='VL_C', bus2_id='Bus_C',
                     b1=1e-6, b2=1e-6, g1=0, g2=0, r=0.5, x=10)

In [ ]:
# Add generator at bus A
network.create_generators(id='GEN_A',
                          voltage_level_id='VL_A',
                          bus_id='Bus_A',
                          target_p=100,
                          min_p=0,
                          max_p=200,
                          target_v=400,
                          voltage_regulator_on=True)
network.create_generators(id='GEN_B',
                          voltage_level_id='VL_B',
                          bus_id='Bus_B',
                          target_p=0,
                          min_p=0,
                          max_p=200,
                          target_v=400,
                          voltage_regulator_on=True)
network.create_generators(id='GEN_C',
                          voltage_level_id='VL_C',
                          bus_id='Bus_C',
                          target_p=0,
                          min_p=0,
                          max_p=200,
                          target_v=400,
                          voltage_regulator_on=True)
# Add loads at B and C
network.create_loads(id='LOAD-1', voltage_level_id='VL_A', bus_id='Bus_A', p0=0, q0=0)
network.create_loads(id='LOAD-2', voltage_level_id='VL_B', bus_id='Bus_B', p0=0, q0=0)
network.create_loads(id='LOAD-3', voltage_level_id='VL_C', bus_id='Bus_C', p0=100, q0=3)



Result

In [ ]:
# Run power flow
print("Running DC Power Flow...")
pp.loadflow.run_dc(network)

In [ ]:
# Display results
print("\nLine Flows:")
print(network.get_lines()[['p1', 'p2']])

In [ ]:
network.get_network_area_diagram()

Sensitivity matrix

In [ ]:
analysis = pp.sensitivity.create_dc_analysis()
analysis.add_branch_flow_factor_matrix(branches_ids=['LINE-AB', 'LINE-AC','LINE-BC'], variables_ids=['LOAD-2'])
result = analysis.run(network)
result.get_reference_matrix()

In [ ]:
result.get_sensitivity_matrix()


In [ ]:
# ============================================================
# Step 2: Create sensitivity analysis
# ============================================================
print("Step 2: Setting up sensitivity analysis...")

sensitivity_analysis = pp.sensitivity.create_dc_analysis()

# Your actual line IDs
lines = ['LINE-AB', 'LINE-AC', 'LINE-BC']

# Your actual injection IDs (generator)
injections = ['GEN_A']

sensitivity_analysis.add_branch_flow_factor_matrix(
    matrix_id='PTDF',
    branches_ids=lines,
    variables_ids=injections
)

print(f"✓ Added sensitivity factors\n")

# ============================================================
# Step 3: Run sensitivity analysis
# ============================================================
print("Step 3: Running sensitivity analysis...")

results = sensitivity_analysis.run(network)

print("✓ Analysis complete\n")

# ============================================================
# Step 4: Extract PTDF matrix
# ============================================================
print("="*60)
print("PTDF MATRIX")
print("="*60)

ptdf_matrix = results.get_branch_flows_sensitivity_matrix('PTDF')
print("\nRows = Lines, Columns = Injections")
print("Value = MW change in line flow per 1 MW injection change\n")
print(ptdf_matrix)

In [ ]:
network.get_single_line_diagram('VL_A')
